[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/cs377/blob/main/supplemental/Fraud_Pair.ipynb)

# Rare Fraud: Which Numbers Still Tell You Something?
**CS 377 · companion to 18_1_5 Credit Card Fraud and 18_1_6 Nested CV · pair activity**

**Driver:** this notebook is yours. Your partner, the **Recorder**, has the **Rare Fraud quiz in Canvas** open, reads each question aloud and types your agreed answers.

**How the two fit together.** The quiz is the guide; this notebook is where you do the work for the quiz questions tagged **💻 NOTEBOOK**. Each section below is named after its quiz question (**Q3** here is **Q3** in the quiz). Questions tagged **💬 NO NOTEBOOK** have no section here: just talk them through.

**For each notebook question:** the code is written for you. The Recorder reads the question aloud; you find the matching section, make the small change marked **✏️** and run the cell (Shift+Enter). Read the result aloud and agree with your partner on what it means. Hints are under each section: try without them first, and open Hint 1 before Hint 2.

## Setup (run this first)
**Part 1 of the quiz (Q1-Q2) has no notebook work.** Run this cell while the Recorder reads Part 1. It downloads 284,807 transactions and trains six models (the 18_1_5 model plus five cross-validation models), so give it a little while.

It rebuilds the 18_1_5 model (the same data, the same split, XGBoost **without** `scale_pos_weight`) and makes what every section uses:
- `X_train`, `X_test`, `y_train`, `y_test`: 199,364 training and 85,443 test transactions; in `y`, 1 means fraud and 0 means legitimate.
- `y_proba`: the model's predicted probability of fraud for each test transaction.
- `oof_proba`: out-of-fold probabilities for the training transactions, computed with `cross_val_predict` as in 18_1_5.
- `make_model(weight)`: the same XGBoost model with `scale_pos_weight=weight`, not yet trained.
- `CLASS_RATIO`: legitimate transactions per fraud in the training set (about 578.5). `COST_FP = 100` and `COST_FN = 450` are the 18_1_5 costs.
- `summary(y_true, proba, threshold)`: recall, precision, false alarms, missed fraud and total cost at a threshold.
- `cost_optimal_threshold(y_true, proba, cost_fp, cost_fn)`: the threshold with the lowest total cost. 18_1_5 tried 100 evenly spaced thresholds from 0.01 to 0.99; this function tries every distinct probability, so it can go below 0.01 (Q5 needs that). For the 18_1_5 costs it picks 0.086 instead of 18_1_5's 0.099, which changes the test cost by $300.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score

pd.set_option('display.width', 140)

# Credit card fraud, prepared exactly as in 18_1_5
df = fetch_openml(name='creditcard', version=1, as_frame=True).frame
df['Class'] = df['Class'].astype(int)          # 1 = fraud, 0 = legitimate
X = df.drop(columns=['Class'])
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

CLASS_RATIO = (y_train == 0).sum() / (y_train == 1).sum()   # legitimate per fraud in training, about 578.5
COST_FP, COST_FN = 100, 450                                  # the 18_1_5 costs


def make_model(weight=1):
    """The 18_1_5 XGBoost model, with scale_pos_weight set to `weight` (18_1_5 uses weight 1)."""
    return xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, scale_pos_weight=weight,
                             eval_metric='aucpr', random_state=42)


def summary(y_true, proba, threshold):
    """Recall, precision, false alarms, missed fraud and total cost ($100 per FP, $450 per FN) at a threshold."""
    y_true = np.asarray(y_true)
    pred = np.asarray(proba) >= threshold
    tp = int((pred & (y_true == 1)).sum())
    fp = int((pred & (y_true == 0)).sum())
    fn = int((~pred & (y_true == 1)).sum())
    return {'recall': tp / (tp + fn), 'precision': tp / (tp + fp) if tp + fp else 0.0,
            'false alarms': fp, 'missed fraud': fn, 'cost': fp * COST_FP + fn * COST_FN}


def cost_optimal_threshold(y_true, proba, cost_fp, cost_fn):
    """The threshold with the lowest total cost on (y_true, proba), trying every distinct probability.
    18_1_5 tried 100 evenly spaced thresholds from 0.01 to 0.99; this search can also go below 0.01."""
    fpr, tpr, thresholds = roc_curve(y_true, proba, drop_intermediate=False)
    n_fraud, n_legit = (np.asarray(y_true) == 1).sum(), (np.asarray(y_true) == 0).sum()
    total_cost = (1 - tpr) * n_fraud * cost_fn + fpr * n_legit * cost_fp
    return thresholds[np.argmin(total_cost)]


model = make_model(1).fit(X_train, y_train)     # the 18_1_5 model: no scale_pos_weight
y_proba = model.predict_proba(X_test)[:, 1]     # P(fraud) for the test transactions
oof_proba = cross_val_predict(make_model(1), X_train, y_train, cv=5,
                              method='predict_proba')[:, 1]   # out-of-fold P(fraud) for the training transactions

print(f"training transactions: {len(y_train):,} ({y_train.sum()} fraud)   "
      f"test transactions: {len(y_test):,} ({y_test.sum()} fraud)")

# Part 2 (Q3-Q6): The 18_1_5 Model

## Q3 · What the ROC curve hides
> **Quiz Q3:** The Q3 cell walks along the test ROC curve (AUC 0.973) and stops at the first threshold that reaches each recall target, then reports the FPR, the number of false alarms and the precision there. Right now it has only one target. Change `TARGETS` to `[0.80, 0.85, 0.90, 0.95]` and run it. Going from 80% to 90% recall, how much does the FPR rise (as a difference, which is the distance the point moves along the ROC curve's x-axis), and how many false alarms does that add? What happens to precision? Use Q1(c) to explain why a curve with AUC 0.973 makes this look like a small price.

**✏️ Your change:** Change `TARGETS = [0.80]` to `TARGETS = [0.80, 0.85, 0.90, 0.95]`, then run the cell.

**Tell your Recorder:** the rise in FPR and in false alarms from 80% to 90% recall, then why the ROC curve hides the cost.

<details><summary>Hint 1</summary>

`TARGETS` is a list of recall values. The cell adds one row to the table for each value.

</details>

<details><summary>Hint 2</summary>

`TARGETS = [0.80, 0.85, 0.90, 0.95]`

</details>

In [ ]:
TARGETS = [0.80]      # ✏️ try [0.80, 0.85, 0.90, 0.95]

fpr, tpr, roc_thresholds = roc_curve(y_test, y_proba)
n_fraud, n_legit = (y_test == 1).sum(), (y_test == 0).sum()
rows = []
for target in TARGETS:
    i = np.argmax(tpr >= target)          # first point on the curve that reaches the target
    tp, fp = round(tpr[i] * n_fraud), round(fpr[i] * n_legit)
    rows.append({'recall target': target, 'recall reached': tpr[i], 'threshold': roc_thresholds[i],
                 'FPR': fpr[i], 'false alarms': fp, 'precision': tp / (tp + fp)})

print(f'test ROC AUC: {roc_auc_score(y_test, y_proba):.3f}')
pd.DataFrame(rows).set_index('recall target').round(4)

## Q4 · What F-beta asks you to decide
> **Quiz Q4:** 18_1_5 picked its threshold by maximizing F-beta with β = 2 on the out-of-fold probabilities. The Q4 cell does the same, then reports how that threshold does on the test set. Change `BETAS` to `[0.5, 1, 2]` and run it. As β rises, which way does the threshold move, and what is β a statement about? β = 2 lands on 0.086, which is also the threshold the $450 / $100 costs pick in Q5. Without knowing the costs, how would you have known that β = 2, and not 1 or 0.5, was the right choice?

**✏️ Your change:** Change `BETAS = [2]` to `BETAS = [0.5, 1, 2]`, then run the cell.

**Tell your Recorder:** which way the threshold moves as β rises, what β expresses, then how you could know which β to use.

<details><summary>Hint 1</summary>

`BETAS` is a list. The cell finds one threshold for each β and adds a row to the table.

</details>

<details><summary>Hint 2</summary>

`BETAS = [0.5, 1, 2]`

</details>

In [ ]:
BETAS = [2]      # ✏️ try [0.5, 1, 2]

precision, recall, pr_thresholds = precision_recall_curve(y_train, oof_proba)
rows = []
for beta in BETAS:
    f_beta = (1 + beta**2) * precision * recall / (beta**2 * precision + recall)
    best_t = pr_thresholds[np.argmax(np.nan_to_num(f_beta[:-1]))]   # chosen on training data only
    rows.append({'beta': beta, 'threshold': best_t, **summary(y_test, y_proba, best_t)})

pd.DataFrame(rows).set_index('beta').round(3)

## Q5 · The cost ratio hidden in Youden's J
> **Quiz Q5:** In 18_1_5, Youden's J chose a threshold of 0.0013, and on the test set it cost more than ten times as much as the default threshold. The Q5 cell prints J's threshold, then finds the cost-optimal threshold on the out-of-fold probabilities for each ratio of FN cost to FP cost in `RATIOS`. Right now it tries only 18_1_5's ratio, 4.5. Add `100` and `CLASS_RATIO` to the list and run it. (This search tries every distinct probability, so for ratio 4.5 it picks 0.086 rather than 18_1_5's 0.099, and its test cost is $300 higher than Q2's.) Which ratio gives exactly J's threshold, and where does that number come from? 18_1_5 says J treats a false positive and a false negative as equally costly. That is true of the rates: J counts one point of TPR the same as one point of FPR. Finish this sentence, counting transactions instead of rates: "Youden's J acts as if one missed fraud costs as much as ... false alarms." Why does that make J a poor choice for this bank?

**✏️ Your change:** Change `RATIOS = [4.5]` to `RATIOS = [4.5, 100, CLASS_RATIO]`, then run the cell.

**Tell your Recorder:** which ratio matches J's threshold and the test cost there, your sentence about J, and why it is a poor choice.

<details><summary>Hint 1</summary>

`CLASS_RATIO` is already defined in Setup. Put it in the list like any other number.

</details>

<details><summary>Hint 2</summary>

`RATIOS = [4.5, 100, CLASS_RATIO]`

</details>

In [ ]:
RATIOS = [4.5]      # ✏️ add 100 and CLASS_RATIO

fpr, tpr, roc_thresholds = roc_curve(y_train, oof_proba)
j_threshold = roc_thresholds[np.argmax(tpr - fpr)]
print(f"Youden's J threshold (out-of-fold): {j_threshold:.4f}")
print(f'CLASS_RATIO = {(y_train == 0).sum():,} legitimate / {(y_train == 1).sum()} fraud = {CLASS_RATIO:.1f}')

rows = []
for ratio in RATIOS:
    best_t = cost_optimal_threshold(y_train, oof_proba, cost_fp=1, cost_fn=ratio)
    rows.append({'FN : FP cost ratio': round(ratio, 1), 'threshold': best_t,
                 **summary(y_test, y_proba, best_t)})   # test cost uses the real $450 / $100

pd.DataFrame(rows).set_index('FN : FP cost ratio').round(4)

## Q6 · Scoring models at a fixed threshold
> **Quiz Q6:** 18_1_6 warns against ranking models in a grid search by their cost at a fixed threshold (its `total_cost_scorer` used 0.95). The Q6 cell trains two models, one with `scale_pos_weight=1` and one with `scale_pos_weight=CLASS_RATIO`, and shows each one's test cost at 0.95 and at its own cost-optimal threshold (found on its own out-of-fold probabilities). The average precision (PR-AUC) column is missing: replace `np.nan` with `average_precision_score(y_test, p)` and run it. Which model would a search that scores cost at 0.95 pick? Which does average precision prefer? Which is actually cheaper once each model gets its own threshold? Use the own threshold column to explain why the 0.95 score was misleading. (The mean P(fraud) column helps too: about 0.17% of the transactions really are fraud.)

**✏️ Your change:** Replace `np.nan` with `average_precision_score(y_test, p)`.

**Tell your Recorder:** which model each of the three columns (cost at 0.95, average precision, cost at own threshold) favors, then why the 0.95 score misled.

<details><summary>Hint 1</summary>

`average_precision_score` (imported in Setup) takes the true labels and the probabilities, like `roc_auc_score`. Inside the loop the test probabilities are called `p`.

</details>

<details><summary>Hint 2</summary>

`ap = average_precision_score(y_test, p)`

</details>

In [ ]:
rows = []
for w in [1, CLASS_RATIO]:
    p = make_model(w).fit(X_train, y_train).predict_proba(X_test)[:, 1]
    oof = cross_val_predict(make_model(w), X_train, y_train, cv=5, method='predict_proba')[:, 1]
    own_t = cost_optimal_threshold(y_train, oof, COST_FP, COST_FN)
    ap = np.nan      # ✏️ average precision of p on the test set
    rows.append({'scale_pos_weight': round(w, 1), 'mean P(fraud)': p.mean(),
                 'cost at 0.95': summary(y_test, p, 0.95)['cost'],
                 'average precision': ap, 'own threshold': own_t,
                 'cost at own threshold': summary(y_test, p, own_t)['cost']})

pd.DataFrame(rows).set_index('scale_pos_weight').round(4)

# Extra credit (Q7-Q8): Weighting by Cost, and 18_1_6's Results
Only if you have finished Q1-Q6. Q8 is in the quiz only.

## ⭐ Q7 · Weighting by the cost ratio
> **Quiz Q7:** 18_1_6 says setting `scale_pos_weight` to the class ratio (about 578) would over-weight fraud by about two orders of magnitude relative to the costs, and that `scale_pos_weight=4.5` gives a weighting very similar to its cost-weighted objective. The Q7 cell compares weight 1 with a second weight, but right now both are 1. Change the second to `4.5` and run it. Where does each model's cost-optimal threshold land, and how do their costs at 0.5 and at their own thresholds compare? Why does weighting by the cost ratio move the best threshold toward 0.5? (If one cost is a little lower at 0.5 than at the model's own threshold, that is because the own threshold was chosen on the training data, not the test set.)

**✏️ Your change:** Change the second weight in the line marked ✏️ from `1` to `4.5`, then run the cell.

**Tell your Recorder:** each model's own threshold, its cost at 0.5 and at its own threshold, then why the threshold moved.

<details><summary>Hint 1</summary>

The weight list has two entries; only the second one changes.

</details>

<details><summary>Hint 2</summary>

`weights = [1, 4.5]`

</details>

In [ ]:
weights = [1, 1]      # ✏️ change the second weight to 4.5

rows = []
for w in weights:
    p = make_model(w).fit(X_train, y_train).predict_proba(X_test)[:, 1]
    oof = cross_val_predict(make_model(w), X_train, y_train, cv=5, method='predict_proba')[:, 1]
    own_t = cost_optimal_threshold(y_train, oof, COST_FP, COST_FN)
    rows.append({'scale_pos_weight': w, 'own threshold': own_t,
                 'cost at 0.5': summary(y_test, p, 0.5)['cost'],
                 'cost at own threshold': summary(y_test, p, own_t)['cost']})

pd.DataFrame(rows).set_index('scale_pos_weight').round(4)